In [ ]:
# pip install langgraph
# export OPENAI_API_KEY=...

from typing_extensions import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    topic: str
    research: NotRequired[str]
    summary: NotRequired[str]
    approved: NotRequired[bool]

def research_node(state: State) -> dict:
    # Replace with real RAG/web search later
    topic = state["topic"]
    return {"research": f"Key points about {topic}: (1) definition (2) use-cases (3) risks."}

def summarize_node(state: State) -> dict:
    r = state["research"]
    return {"summary": f"Summary: {r} -> In short, {state['topic']} is useful but needs guardrails."}

def approval_node(state: State) -> Command:
    decision = interrupt(
        {
            "question": "Approve this summary? (yes/no)",
            "topic": state["topic"],
            "summary": state["summary"],
            "expected_format": {"approved": "yes|no"},
        }
    )
    approved = str(decision.get("approved", "")).strip().lower() == "yes"
    return Command(goto="save", update={"approved": approved})

def save_node(state: State) -> dict:
    if not state.get("approved"):
        return {"result": "Not approved — nothing saved."}

    path = "output.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"TOPIC: {state['topic']}\n\n{state['summary']}\n")
    return {"result": f"Saved to {path}"}

builder = StateGraph(State)
builder.add_node("research", research_node)
builder.add_node("summarize", summarize_node)
builder.add_node("approve", approval_node)
builder.add_node("save", save_node)

builder.add_edge(START, "research")
builder.add_edge("research", "summarize")
builder.add_edge("summarize", "approve")
builder.add_edge("save", END)

graph = builder.compile(checkpointer=InMemorySaver())

if __name__ == "__main__":
    cfg = {"configurable": {"thread_id": "t1"}}

    # 1) First run -> interrupts
    out = graph.invoke({"topic": "Agentic AI"}, config=cfg)
    print("INTERRUPT PAYLOAD:\n", out["__interrupt__"])

    # 2) Resume with human decision
    out2 = graph.invoke(Command(resume={"approved": "yes"}), config=cfg)
    print("FINAL:\n", out2)

In [ ]:
# pip install autogen-agentchat~=0.2
# export OPENAI_API_KEY=...

import os

from autogen import AssistantAgent, UserProxyAgent, register_function

llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",  # change as needed
            "api_key": os.environ["OPENAI_API_KEY"],
        }
    ]
}

def save_to_file(topic: str, summary: str, path: str = "output.txt") -> str:
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"TOPIC: {topic}\n\n{summary}\n")
    return f"Saved to {path}"

researcher = AssistantAgent(
    name="researcher",
    llm_config=llm_config,
    system_message="You research topics. Return bullet points with key facts.",
)

writer = AssistantAgent(
    name="writer",
    llm_config=llm_config,
    system_message="You write concise summaries based on research. Keep it short.",
)

# Human in the loop: ALWAYS asks user for input at each step (simple demo)
human = UserProxyAgent(
    name="human",
    human_input_mode="ALWAYS",
    code_execution_config=False,
)

# Tool wiring: writer will call, human will execute
register_function(
    save_to_file,
    caller=writer,
    executor=human,
    name="save_to_file",
    description="Save the approved summary to a text file.",
)

if __name__ == "__main__":
    topic = "Agentic AI"

    # Step 1: Researcher produces research
    r = human.initiate_chat(
        researcher,
        message=f"Research '{topic}'. Give 5 bullets.",
        max_turns=1,
    )
    research_text = r.summary or r.chat_history[-1]["content"]

    # Step 2: Writer drafts summary
    w = human.initiate_chat(
        writer,
        message=f"Write a 6-8 line summary on '{topic}' using:\n{research_text}",
        max_turns=1,
    )
    summary_text = w.summary or w.chat_history[-1]["content"]

    # Step 3: Human approval
    print("\n--- DRAFT SUMMARY ---\n", summary_text)
    approval = input("\nApprove? (yes/no): ").strip().lower()

    if approval == "yes":
        # Step 4: Ask writer to save using tool
        human.initiate_chat(
            writer,
            message=(
                f"Approved. Call tool save_to_file with topic='{topic}' "
                f"and summary='''{summary_text}''' and path='output.txt'."
            ),
            max_turns=2,
        )
    else:
        print("Not approved — nothing saved.")

In [ ]:
researcher:
  role: "Researcher"
  goal: "Research the topic and return key bullets."
  backstory: "You are thorough and concise."

writer:
  role: "Writer"
  goal: "Write a short summary from the research."
  backstory: "You write clear, structured summaries."

research_task:
  description: "Research {topic} and provide 5 bullet points."
  expected_output: "5 bullet points with key facts."
  agent: researcher

summary_task:
  description: "Write a 6-8 line summary for {topic} using the research."
  expected_output: "A concise summary."
  agent: writer

# pip install crewai
from crewai import Agent, Task, Crew, Process
import yaml

def load_yaml(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def build_crew(topic: str) -> Crew:
    agents_cfg = load_yaml("agents.yaml")
    tasks_cfg = load_yaml("tasks.yaml")

    researcher = Agent(
        role=agents_cfg["researcher"]["role"],
        goal=agents_cfg["researcher"]["goal"],
        backstory=agents_cfg["researcher"]["backstory"],
        verbose=True,
    )

    writer = Agent(
        role=agents_cfg["writer"]["role"],
        goal=agents_cfg["writer"]["goal"],
        backstory=agents_cfg["writer"]["backstory"],
        verbose=True,
    )

    research_task = Task(
        description=tasks_cfg["research_task"]["description"].format(topic=topic),
        expected_output=tasks_cfg["research_task"]["expected_output"],
        agent=researcher,
    )

    summary_task = Task(
        description=tasks_cfg["summary_task"]["description"].format(topic=topic),
        expected_output=tasks_cfg["summary_task"]["expected_output"],
        agent=writer,
        context=[research_task],
    )

    return Crew(
        agents=[researcher, writer],
        tasks=[research_task, summary_task],
        process=Process.sequential,
        verbose=True,
    )

from crew import build_crew

def save_to_file(topic: str, summary: str, path: str = "output.txt") -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"TOPIC: {topic}\n\n{summary}\n")

if __name__ == "__main__":
    topic = "Agentic AI"
    crew = build_crew(topic)

    result = crew.kickoff()
    summary_text = str(result)

    print("\n--- DRAFT SUMMARY ---\n", summary_text)
    approval = input("\nApprove? (yes/no): ").strip().lower()

    if approval == "yes":
        save_to_file(topic, summary_text, "output.txt")
        print("Saved to output.txt")
    else:
        print("Not approved — nothing saved.")

In [ ]:
from openai import OpenAI
from openai.types.beta.threads import Thread
import os

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# 1️Create Assistant
assistant = client.beta.assistants.create(
    name="Research Assistant",
    instructions="Research a topic, write a short summary.",
    model="gpt-4o-mini",
)

# 2️Create Thread
thread = client.beta.threads.create()

topic = "Agentic AI"

# Add user message
client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content=f"Research {topic} and give a 6-8 line summary.",
)

# Run assistant
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id,
    assistant_id=assistant.id,
)

messages = client.beta.threads.messages.list(thread_id=thread.id)

summary_text = messages.data[0].content[0].text.value

print("\n--- SUMMARY ---\n", summary_text)

# Human Approval
approval = input("\nApprove? (yes/no): ").strip().lower()

if approval == "yes":
    with open("output.txt", "w") as f:
        f.write(summary_text)
    print("Saved to output.txt")
else:
    print("Not approved.")

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import tool
import os

@tool
def save_to_file(summary: str):
    with open("output.txt", "w") as f:
        f.write(summary)
    return "Saved successfully"

agent = Agent(
    model="gemini-1.5-pro",
    instructions="Research topic and write short summary.",
    tools=[save_to_file],
)

topic = "Agentic AI"

# Run agent
response = agent.run(
    f"Research {topic} and give 6-8 line summary."
)

summary_text = response.output_text

print("\n--- SUMMARY ---\n", summary_text)

# Human approval
approval = input("\nApprove? (yes/no): ").strip().lower()

if approval == "yes":
    agent.run(
        f"Call tool save_to_file with summary: {summary_text}"
    )
else:
    print("Not approved.")